# 04 — Fine-tuning BETO (3 clases) en Google Colab (#36)

**Qué hace:** entrena un modelo en español que clasifica un mensaje (ya anonimizado) en:

| Clase | Semáforo | Significado |
|---|---|---|
| `none` | GREEN | legítimo |
| `spam` | AMBER | publicidad o molestia |
| `fraud` | RED | estafa |

**Cómo usarlo:**
1. En Colab: **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**.
2. **Entorno de ejecución → Ejecutar todo**.
3. Cuando lo pida, sube el CSV de SpaPhish y el CSV de revisión de SpamSpa.
4. Al terminar se descarga `avizor-3c.zip` con el modelo y `metrics.json`.

Los datos subidos se borran al cerrar la sesión de Colab. No se suben a GitHub.

**1. Comprobamos que hay GPU.**

In [ ]:
import torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO (cambia el entorno a GPU)")

**2. Descargamos el código de la rama del #36 e instalamos lo que falta.**

In [ ]:
BRANCH = "feat/36-finetune-beto"

!git clone -q -b {BRANCH} https://github.com/MariaIsaDurango/Avizor.git
%cd Avizor/ml
!pip install -q sentencepiece "accelerate>=1.1" "datasets>=2.20" "transformers>=4.46"

**3. Descargamos SpamSpa de HuggingFace** (el `train.csv`; el `test.csv` del dataset está roto).

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

RAW = Path("data/raw")
(RAW / "spamspa").mkdir(parents=True, exist_ok=True)
(RAW / "spaphish").mkdir(parents=True, exist_ok=True)

spamspa_path = hf_hub_download(
    repo_id="softecapps/spam_ham_spanish",
    filename="train.csv",
    repo_type="dataset",
    local_dir=RAW / "spamspa",
)
print("SpamSpa:", spamspa_path)

**4. Subimos los dos CSV locales:**
- `Spaphish dataset - DiB.csv`
- `spamspa_review.csv` (la revisión de publicidad / estafa)

In [ ]:
from google.colab import files

uploaded = files.upload()
for name, content in uploaded.items():
    target = RAW / ("spaphish" if "paphish" in name.lower() else "spamspa") / name
    target.write_bytes(content)
    print("Guardado:", target)

SPAPHISH = next((RAW / "spaphish").glob("*.csv"))
REVIEW = next((RAW / "spamspa").glob("*review*.csv"))
print(SPAPHISH, REVIEW, sep="\n")

**5. Entrenamos.** Con GPU T4 tarda unos 15 minutos.

Usa BETO (`dccuchile/bert-base-spanish-wwm-cased`) con peso 3 para la clase fraud.

In [ ]:
!python -m src.classifier.train \
    --spaphish "{SPAPHISH}" \
    --spamspa "{spamspa_path}" \
    --review "{REVIEW}" \
    --out models/avizor-3c \
    --epochs 5

**6. Resultados.** Los dos objetivos del #36 aparecen en `targets`.

In [ ]:
import json
from pathlib import Path

metrics = json.loads(Path("models/avizor-3c/metrics.json").read_text())
print("Modelo base:", metrics["base_model"], "| versión:", metrics["model_version"])
print("Objetivos:", metrics["targets"])
print("Latencia CPU (mediana):", round(metrics["cpu_latency_ms_median"]), "ms")
for source, values in metrics["test_by_source"].items():
    print(f"\n{source}:", {k: round(v, 3) for k, v in values.items() if isinstance(v, float)})

**7. Descargamos el modelo y las métricas** (sin los CSV ni los checkpoints).

In [ ]:
!cd models/avizor-3c && zip -qr /content/avizor-3c.zip model metrics.json
files.download("/content/avizor-3c.zip")